# Health keywords

## Setup & configuration

In [ ]:
import os
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

# === Paths: adjust if needed ===
BASE_DIR = Path("../data/papers/neurips2024_health")  # folder from previous notebook
CSV_ALL = BASE_DIR / "neurips2024_all_from_neurips_site.csv"
CSV_HEALTH = BASE_DIR / "neurips2024_health_from_neurips_site.csv"
CSV_HEALTH_WITH_STATUS = BASE_DIR / "neurips2024_health_with_pdf_status.csv"  # optional
PDF_DIR = BASE_DIR / ""  # where PDFs were downloaded

TEXT_CACHE = BASE_DIR / "neurips2024_health_with_text.pkl"  # cache full-text so you don't re-parse PDFs
ASSOC_OUT = BASE_DIR / "neurips2024_health_physionet_assoc.csv"

print("Base directory:", BASE_DIR.resolve())
print("PDF directory :", PDF_DIR.resolve())

## Load CSVs

In [ ]:
# Load all papers (optional; can be useful for denominators)
all_df = pd.read_csv(CSV_ALL)
print("All papers:", len(all_df))

# Load health subset
health_df = pd.read_csv(CSV_HEALTH)
print("Health-related papers:", len(health_df))

# If you have a CSV with pdf_download_status, load and use that instead
if CSV_HEALTH_WITH_STATUS.exists():
    print("Found health_with_pdf_status CSV; using that.")
    health_df = pd.read_csv(CSV_HEALTH_WITH_STATUS)
    print("Health-related papers (with status):", len(health_df))

health_df.head()

## Define vocabularies

In [ ]:
# Health-related keywords (for context; you already used some version of this)
HEALTH_TEXT_KEYWORDS = [
    "health", "healthcare", "health care",
    "medical", "medicine", "clinical",
    "hospital", "ward",
    "patient", "patients",
    "electronic health record", "ehr",
    "intensive care", "icu",
    "radiology", "ct", "mri", "x-ray", "xray",
    "biomedical", "bio-medical",
    "bioinformatics",
    "genomic", "genomics",
    "proteomic", "protein",
    "drug", "pharmacology",
    "therapeutic", "therapy",
    "disease", "diagnosis", "screening",
    "epidemiology", "public health",
    "covid", "sepsis", "mortality",
    "clinical trial", "trial",
    "ecg", "electrocardiogram",
    "eeg", "electroencephalogram",
]

# Searching for lower case terms
# So use lowercase only
PHYSIONET_TERMS = [
            "physionet",
            "mimic-ii",
            "mimic-iv",
            "mimic-ed",
            "mimic-cxr",
            "mimic-iv-ed",
            "mimic-iv-cxr",
            "mimic-iv-ecg",
            "mimic-iv-echo",
            "wfdb",
            "10.13026",
            "eicu-crd",
            "mit-bih",
           ]

print("Health keywords:", len(HEALTH_TEXT_KEYWORDS))
print("PhysioNet-related terms:", len(PHYSIONET_TERMS))


In [ ]:
def sanitize_filename(name: str, max_len: int = 120) -> str:
    keep = "-_.() "
    name = "".join(c if c.isalnum() or c in keep else "_" for c in name)
    if len(name) > max_len:
        name = name[:max_len]
    return name.strip(" _")


## Extract full text from PDFs (with caching)

In [ ]:
from pypdf import PdfReader

def extract_pdf_text(path: Path) -> str:
    try:
        reader = PdfReader(str(path))
        texts = []
        for page in reader.pages:
            try:
                txt = page.extract_text() or ""
            except Exception:
                txt = ""
            texts.append(txt)
        return "\n".join(texts)
    except Exception as e:
        # You could log e here
        return ""

if 1==2:
    print("Found cached text at:", TEXT_CACHE)
    health_with_text = pd.read_pickle(TEXT_CACHE)
    print("Loaded cached text for", len(health_with_text), "health papers.")
else:
    print("No cache found; extracting text from PDFs (this may take a while).")

    texts = []
    pdf_status = []  # in case we want to track success/failure differently from pdf_download_status

    for _, row in tqdm(health_df.iterrows(), total=len(health_df), desc="Reading PDFs"):
        title = row.get("title") or ""
        sanitized = sanitize_filename(title) or "paper"
        pdf_path = PDF_DIR / f"{sanitized}.pdf"

        # Build a base text from title + abstract (so we have *something* even if no PDF)
        title_ = (row.get("title") or "")
        abstract_ = (row.get("abstract") or "")
        base_text = f"{title_}\n\n{abstract_}"

        if pdf_path.exists():
            pdf_text = extract_pdf_text(pdf_path)
            full_text = base_text + "\n\n" + pdf_text
            pdf_status.append("pdf_ok")
        else:
            full_text = base_text
            pdf_status.append("pdf_missing")
            import pdb; pdb.set_trace()

        texts.append(full_text)

    health_with_text = health_df.copy()
    health_with_text["full_text"] = texts
    health_with_text["pdf_read_status_local"] = pdf_status

    # Normalize to lowercase for keyword matching
    health_with_text["full_text_norm"] = health_with_text["full_text"].str.lower()

    health_with_text.to_pickle(TEXT_CACHE)
    print("Saved cached text to:", TEXT_CACHE)

health_with_text[["title", "pdf_read_status_local"]].head()


## Build keyword matrix

In [ ]:
df = health_with_text.copy()
print("Health papers with text:", len(df))

def add_keyword_columns(df, keywords, prefix=None):
    for term in keywords:
        col = term if prefix is None else f"{prefix}__{term}"
        df[col] = df["full_text_norm"].str.contains(term, regex=False).astype(int)
    return df

df = add_keyword_columns(df, HEALTH_TEXT_KEYWORDS, prefix="H")
df = add_keyword_columns(df, PHYSIONET_TERMS, prefix="P")

df.head()


## Build matrix

In [ ]:
results = []

N = len(df)

for h in HEALTH_TEXT_KEYWORDS:
    col_h = f"H__{h}"
    if col_h not in df.columns:
        continue

    count_h = df[col_h].sum()
    if count_h == 0:
        continue  # no papers with this keyword

    for p in PHYSIONET_TERMS:
        col_p = f"P__{p}"
        if col_p not in df.columns:
            continue

        count_p = df[col_p].sum()
        if count_p == 0:
            continue

        # co-occurrence
        co = (df[col_h] & df[col_p]).sum()
        if co == 0:
            continue  # skip pairs that never co-occur

        # probabilities
        p_h = count_h / N
        p_p = count_p / N
        p_hp = co / N

        p_p_given_h = co / count_h
        p_h_given_p = co / count_p

        lift = p_hp / (p_h * p_p) if p_h > 0 and p_p > 0 else None

        results.append({
            "health_keyword": h,
            "physionet_term": p,
            "count_H": int(count_h),
            "count_P": int(count_p),
            "cooccurrence": int(co),
            "P(P|H)": p_p_given_h,
            "P(H|P)": p_h_given_p,
            "lift": lift,
        })

assoc_df = pd.DataFrame(results)
print("Association rows:", len(assoc_df))

# Sort by lift (strongest associations first)
assoc_df_sorted = assoc_df.sort_values(["lift", "cooccurrence"], ascending=[False, False])
assoc_df_sorted.head(20)


## Save matrix

In [ ]:
ASSOC_OUT.parent.mkdir(exist_ok=True, parents=True)
assoc_df_sorted.to_csv(ASSOC_OUT, index=False)
print("Saved association table to:", ASSOC_OUT)

# Top health keywords most strongly associated with PhysioNet-related terms
print("\nTop health keywords by max lift (over any PhysioNet term):")
top_by_health = (
    assoc_df_sorted
    .groupby("health_keyword")["lift"]
    .max()
    .reset_index()
    .sort_values("lift", ascending=False)
)
top_by_health.head(20)


In [ ]:
# Example: which health keywords are most associated with "physionet"?
assoc_df_sorted[assoc_df_sorted["physionet_term"] == "physionet"].head(20)


In [ ]:
# Any physionet-related term present?
physio_cols = [f"P__{t}" for t in PHYSIONET_TERMS if f"P__{t}" in df.columns]
df["any_physionet_term"] = df[physio_cols].max(axis=1)

n_health = len(df)
n_health_physio = df["any_physionet_term"].sum()

print(f"Health-related NeurIPS papers: {n_health}")
print(f"...of which mention at least one PhysioNet/MIMIC/ICU-related term: {n_health_physio}")
print(f"Proportion: {n_health_physio / n_health:.3f}")
